# 04. Context Compression

- 검색 결과가 길거나 문서 수가 많으면 LLM에 전달되는 context가 커진다.
- 이때 
  1. 비용과 지연 시간이 증가한다.
  2. 질문과 직접 관련 없는 정보가 답변에 섞일 수 있다.
  3. 중요한 근거가 긴 context 안에 묻힐 수 있다.
- Context Compression은 검색된 문서를 LLM에 넣기 전에 질문과 관련 있는 내용 중심으로 줄이는 방법이다.

In [1]:
from dotenv import load_dotenv
load_dotenv()

LLM_MODEL = 'gpt-4.1-mini'

In [2]:
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model=LLM_MODEL, temperature=0)
output_parser = StrOutputParser()

## 문서 준비

- C01: 기업 RAG 시스템의 생성 단계 설계와 직접 관련 있음
- C02: AI 교육 플랫폼에 대한 문서로, 일부 표현은 그럴듯하지만 질문과 직접 관련 없음
- C03: AI 채용 심사에 대한 문서로, 질문과 직접 관련 없음

In [3]:
DOCS = [
    Document(
        page_content="""
기업의 사내 지식 검색 시스템에서는 RAG 구조를 사용해 최신 문서와 내부 규정을 바탕으로 답변을 생성한다.
검색 단계에서는 질문과 관련 있는 문서를 찾고, 생성 단계에서는 검색된 문서를 context로 넣어 답변을 만든다.
생성 단계에서는 문서 밖 내용을 추측하지 않도록 프롬프트 규칙을 명확히 작성해야 한다.
또한 사용한 문서 ID나 출처를 함께 표시하면 사용자가 답변의 근거를 확인하기 쉽다.
검색된 문서가 부족하거나 질문과 맞지 않는 경우에는 답변을 생성하지 않고 근거 부족을 알려야 한다.
너무 많은 문서를 한 번에 넣으면 질문과 직접 관련 없는 내용이 답변에 섞일 수 있으므로 context 구성도 함께 관리해야 한다.
""",
        metadata={'doc_id': 'C01', 'title': '기업 RAG 시스템의 생성 단계 설계'}
    ),
    Document(
        page_content="""
인공지능 기반 교육 플랫폼은 학습자의 풀이 시간, 정답률, 오답 유형을 분석하여 개인화된 학습 경로를 제공한다.
챗봇 튜터는 학습자의 질문에 답변하고, 강의 내용을 요약하거나 복습 질문을 생성하는 데 활용된다.
교육 분야에서는 개인정보 보호, 학습 데이터 편향, 과도한 자동화에 따른 교사의 역할 축소 문제를 함께 고려해야 한다.
학생에게 제공되는 피드백은 지나치게 단정적이면 안 되며, 학습자의 현재 수준에 맞게 조정되어야 한다.
""",
        metadata={'doc_id': 'C02', 'title': 'AI 교육 플랫폼의 활용과 주의점'}
    ),
    Document(
        page_content="""
AI 채용 심사 시스템은 이력서, 자기소개서, 면접 기록 등을 분석하여 지원자를 평가하는 데 사용될 수 있다.
이때 성별, 연령, 출신 지역, 학력 같은 정보가 부당하게 판단에 영향을 주지 않도록 공정성을 점검해야 한다.
채용 결과에 대해 설명 가능한 근거를 제공하는 것도 중요하며, 민감 정보 처리 기준을 명확히 해야 한다.
공정성 검증 없이 AI 판단 결과를 그대로 사용하는 것은 위험하다.
""",
        metadata={'doc_id': 'C03', 'title': 'AI 채용 심사의 공정성'}
    ),
]

def fake_retriever(query: str):
    return DOCS


def format_docs(docs):
    return '\n\n'.join(
        f"[{doc.metadata['doc_id']}] {doc.metadata['title']}\n{doc.page_content.strip()}"
        for doc in docs
    )

## 압축하지 않은 Context

검색된 문서를 그대로 넣으면 질문과 직접 관련 없는 문서까지 함께 context에 들어간다.

In [4]:
query = '기업 RAG 시스템의 생성 단계에서 주의할 점은 무엇인가요?'
retrieved_docs = fake_retriever(query)
raw_context = format_docs(retrieved_docs)

print('문서 수:', len(retrieved_docs))
print('context 길이:', len(raw_context))
print(raw_context)

문서 수: 3
context 길이: 889
[C01] 기업 RAG 시스템의 생성 단계 설계
기업의 사내 지식 검색 시스템에서는 RAG 구조를 사용해 최신 문서와 내부 규정을 바탕으로 답변을 생성한다.
검색 단계에서는 질문과 관련 있는 문서를 찾고, 생성 단계에서는 검색된 문서를 context로 넣어 답변을 만든다.
생성 단계에서는 문서 밖 내용을 추측하지 않도록 프롬프트 규칙을 명확히 작성해야 한다.
또한 사용한 문서 ID나 출처를 함께 표시하면 사용자가 답변의 근거를 확인하기 쉽다.
검색된 문서가 부족하거나 질문과 맞지 않는 경우에는 답변을 생성하지 않고 근거 부족을 알려야 한다.
너무 많은 문서를 한 번에 넣으면 질문과 직접 관련 없는 내용이 답변에 섞일 수 있으므로 context 구성도 함께 관리해야 한다.

[C02] AI 교육 플랫폼의 활용과 주의점
인공지능 기반 교육 플랫폼은 학습자의 풀이 시간, 정답률, 오답 유형을 분석하여 개인화된 학습 경로를 제공한다.
챗봇 튜터는 학습자의 질문에 답변하고, 강의 내용을 요약하거나 복습 질문을 생성하는 데 활용된다.
교육 분야에서는 개인정보 보호, 학습 데이터 편향, 과도한 자동화에 따른 교사의 역할 축소 문제를 함께 고려해야 한다.
학생에게 제공되는 피드백은 지나치게 단정적이면 안 되며, 학습자의 현재 수준에 맞게 조정되어야 한다.

[C03] AI 채용 심사의 공정성
AI 채용 심사 시스템은 이력서, 자기소개서, 면접 기록 등을 분석하여 지원자를 평가하는 데 사용될 수 있다.
이때 성별, 연령, 출신 지역, 학력 같은 정보가 부당하게 판단에 영향을 주지 않도록 공정성을 점검해야 한다.
채용 결과에 대해 설명 가능한 근거를 제공하는 것도 중요하며, 민감 정보 처리 기준을 명확히 해야 한다.
공정성 검증 없이 AI 판단 결과를 그대로 사용하는 것은 위험하다.


## 압축하지 않은 Context로 답변하기

답변이 틀리지 않을 수 있지만 context 안에 교육, 채용 문서까지 함께 들어가 있으므로 답변이 넓어지거나 불필요한 주의사항이 섞일 가능성이 있다.

In [5]:
answer_prompt = PromptTemplate.from_template("""
다음 문서를 바탕으로 질문에 답하세요.
문서에 없는 내용은 추측하지 마세요.
답변은 4문장 이내로 작성하세요.

[Context]
{context}

[Question]
{question}

[Answer]
""")

answer_chain = answer_prompt | llm | output_parser

raw_answer = answer_chain.invoke({'context': raw_context, 'question': query})
print(raw_answer)

기업 RAG 시스템의 생성 단계에서는 문서 밖 내용을 추측하지 않도록 프롬프트 규칙을 명확히 작성해야 합니다. 사용한 문서 ID나 출처를 함께 표시하여 답변의 근거를 확인할 수 있게 해야 합니다. 검색된 문서가 부족하거나 질문과 맞지 않으면 답변을 생성하지 않고 근거 부족을 알려야 합니다. 또한 너무 많은 문서를 한 번에 넣지 않도록 context 구성을 관리해야 합니다.


## 질문 기준으로 Context 압축하기

- 압축 단계에서는 모든 문서를 무조건 요약하지 않는다.
- 질문과 직접 관련 있는 문서는 필요한 근거만 남기고, 관련 없는 문서는 제거한다.

In [6]:
compress_prompt = PromptTemplate.from_template("""
다음 문서가 질문에 답하는 데 직접 관련이 있는지 판단하세요.

규칙:
1. 질문에 직접 관련이 없으면 "관련 없음"이라고만 답하세요.
2. 직접 관련이 있으면 질문에 답하는 데 필요한 핵심 근거만 2문장 이내로 요약하세요.
3. 문서에 없는 내용을 추가하지 마세요.

[Question]
{question}

[Document]
{document}

[Compressed]
""")

compress_chain = compress_prompt | llm | output_parser


def compress_or_drop_docs(docs, question: str):
    compressed_docs = []
    for doc in docs:
        compressed = compress_chain.invoke({
            'question': question,
            'document': doc.page_content,
        }).strip()

        if compressed == '관련 없음':
            continue

        compressed_docs.append(
            Document(page_content=compressed, metadata=doc.metadata)
        )
    return compressed_docs

compressed_docs = compress_or_drop_docs(retrieved_docs, query)
compressed_context = format_docs(compressed_docs)

print('압축 전 문서 수:', len(retrieved_docs))
print('압축 후 문서 수:', len(compressed_docs))
print('압축 전 context 길이:', len(raw_context))
print('압축 후 context 길이:', len(compressed_context))
print(compressed_context)

압축 전 문서 수: 3
압축 후 문서 수: 1
압축 전 context 길이: 889
압축 후 context 길이: 167
[C01] 기업 RAG 시스템의 생성 단계 설계
생성 단계에서는 문서 밖 내용을 추측하지 않도록 프롬프트 규칙을 명확히 작성해야 하며, 사용한 문서 ID나 출처를 함께 표시해 답변의 근거를 확인할 수 있도록 해야 한다. 또한 너무 많은 문서를 한 번에 넣지 않고 context 구성을 관리해야 한다.


## 압축된 Context로 답변하기

압축 후에는 질문과 직접 관련 있는 근거만 남기고 답변을 생성한다.

In [7]:
compressed_answer = answer_chain.invoke({'context': compressed_context, 'question': query})
print(compressed_answer)

기업 RAG 시스템의 생성 단계에서는 문서 밖 내용을 추측하지 않도록 프롬프트 규칙을 명확히 작성해야 합니다. 답변 시 사용한 문서 ID나 출처를 함께 표시하여 근거를 명확히 해야 합니다. 또한 한 번에 너무 많은 문서를 넣지 않고 context 구성을 적절히 관리하는 것이 중요합니다. [C01]


## 정리

1. 질문과 직접 관련 없는 문서를 제거할 수 있다.
2. LLM에 전달되는 context 길이를 줄일 수 있다.
3. 최종 답변이 더 간결하고 질문 중심으로 정리될 수 있다.
4. 단, 압축 과정에서 중요한 근거가 사라질 수 있으므로 압축 결과를 검토해야 한다.